# Knowledge distillation into a 17M encoder with 10% of SST-2 labels

Teacher: [Ettin-encoder-400m](https://huggingface.co/jhu-clsp/ettin-encoder-400m). Students: [Ettin-encoder-17m](https://huggingface.co/jhu-clsp/ettin-encoder-17m), and 32m and 68m for the size comparison. All are ModernBERT-architecture encoders from one pretraining recipe ([Weller et al., 2025](https://arxiv.org/abs/2507.11412)) with one tokenizer. Task: GLUE SST-2, binary sentiment; the GLUE validation split (872 sentences) is the test set. Code, plan, and results: [project/](https://github.com/PE51K/itmo-ai-talent-hub-efficient-models-course/tree/main/project).

The notebook runs every run of `run_list()` in `kd.py`, skips runs whose result is already in `results/runs/`, and stops starting new runs after `MAX_HOURS`. All 83 runs and both benchmarks are committed to the repository; with `FROM_REPO = True` the notebook copies them to the output and has nothing left to run. To reproduce runs, set `FROM_REPO = False` and list them in `RUNS`, for example `RUNS = "teacher_p10 hard_p10_s0 kdu_p10_s0"` (their training took 9.4 min on a T4 in the first session; with model downloads and teacher logits we estimate about 15 min).

| Group | Runs | T4 time, min |
|---|---|---|
| 10% table, full labels, label curve, selection, `bench` | 45 | 197 |
| Teacher seeds, KD temperature and weight, student size, `bench_sizes` | 29 | 199 |
| KD from teacher seeds 1 and 2, T = 4 with KL only | 9 | 45 |

Student runs have 3 seeds (`_s0`, `_s1`, `_s2`).

## Settings and resuming

Kaggle: Accelerator GPU T4 x2 (one GPU is used) or P100, Internet on. Click Save Version -> Save & Run All; a saved version runs in the background for up to 12 hours, and its `results/` and `results.zip` are in the Output tab.

To continue after a stopped or failed version, add the previous version's output as input (Add Input -> Your Work -> this notebook) and run again: the resume cell copies every `results/` folder and `results.zip` it finds under `/kaggle/input` into the working directory, and finished runs are skipped.

Colab: Runtime -> Change runtime type -> T4 GPU. To resume, upload `results.zip` to `/content` before running. Download `/content/results.zip` at the end; a Colab session loses its files when it disconnects.

In [ ]:
MAX_HOURS = 11.0 # no run starts after this, Kaggle stops a session at 12 hours
RUNS = "" # space-separated run names from kd.py, empty runs everything
FROM_REPO = True # start from the results committed to the repository; False reruns everything in RUNS

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/content")
REPO = Path("/tmp/repo")
# check=True stops Run All here when there is no internet (Kaggle: Settings -> Internet on)
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/PE51K/itmo-ai-talent-hub-efficient-models-course.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
os.chdir(REPO / "project")
subprocess.run(["pip", "install", "-q", "transformers>=4.48", "datasets"], check=True)
# kd.py writes to project/results, which points to the output directory; committed results go there first
(WORK / "results").mkdir(exist_ok=True)
if Path("results").is_dir() and not Path("results").is_symlink():
    if FROM_REPO:
        shutil.copytree("results", WORK / "results", dirs_exist_ok=True)
    shutil.rmtree("results")
if not Path("results").exists():
    Path("results").symlink_to(WORK / "results")

In [ ]:
import zipfile

for src in Path("/kaggle/input").glob("**/results"):
    if (src / "runs").is_dir():
        shutil.copytree(src, WORK / "results", dirs_exist_ok=True)
        print("copied", src)
zips = list(Path("/kaggle/input").glob("**/results.zip"))
if Path("/content/results.zip").exists():
    zips.append(Path("/content/results.zip"))
for z in zips:
    zipfile.ZipFile(z).extractall(WORK)
    print("unpacked", z)
print(len(list((WORK / "results" / "runs").glob("*.json"))), "runs done before")

In [ ]:
import torch

assert torch.cuda.is_available(), "no GPU: set the accelerator in the notebook settings"
print(torch.cuda.get_device_name(), torch.__version__)

## Runs

The loss and the run names (`teacher`, `hard`, `kd`, `kdu`, subsets and suffixes) are defined in the [README](https://github.com/PE51K/itmo-ai-talent-hub-efficient-models-course/tree/main/project#setup).

In [ ]:
!python kd.py {RUNS} --max-hours {MAX_HOURS}

## Results

`plots.py` averages the seeds of each config (mean ± sample std), writes `results/summary.md` and `results/figures/label_curve.png`.

In [ ]:
!python plots.py

In [ ]:
from IPython.display import Image, display

display(Image(str(WORK / "results" / "figures" / "label_curve.png")))

In [ ]:
shutil.make_archive(str(WORK / "results"), "zip", root_dir=WORK, base_dir="results")